# 7. Fine-mapped causal variants at core-motif sites (Fig. 4a,b)

Do the core motifs mark bases that matter? For 16 GTEx tissues with a matched EpiMap biosample, the fine-mapped causal eQTLs (SuSiE posterior inclusion probability > 0.9, the benchmark released with Enformer) are counted at the sites of the 43 core motifs in the biosample's promoter windows and at the sites of ten random control sets of 43 JASPAR motifs that resemble no core motif. A variant counts when it lies within 4 bp of a site centre. Panel **a** shows the counts, panel **b** the overlap ratio (variants per motif occurrence); the test is a paired one-sided Wilcoxon over the 16 tissues against the mean of the ten control sets.

`code/enformer_snp_overlap/snp_overlap_universal.py` computes the table (about 8 min), `figures/fig4/make_fig4_panels.py` draws the panels. Inputs: Zenodo `fig4_eqtl_inputs.tar.gz` unpacked under `$ENHANCER_ROOT` (the eQTL benchmark, the tissue-to-biosample map, the window coordinates of the 26 eQTL biosamples and their core-motif and control-set FIMO site scans; no FIMO run is needed).

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
SN = f"{CODE}/enformer_snp_overlap"; F4 = f"{REPO}/figures/fig4"; WORK = f"{ENHANCER_ROOT}/notebook_runs/fig4"; os.makedirs(WORK, exist_ok=True)
import numpy as np, pandas as pd
from IPython.display import Image, display
for k, p in {"eQTL benchmark": f"{ENHANCER_ROOT}/enformer_comparison/eqtl-enformer", "tissue -> biosample map": f"{ENHANCER_ROOT}/geuvadis/predixscan/en_models_to_BSSID.csv",
             "window coordinates": f"{ENHANCER_ROOT}/data_generated", "core-motif scans": f"{CODE}/posthoc_analysis/results/fimo_universal", "control-set scans": f"{CODE}/posthoc_analysis/results/fimo_multi/set_0"}.items():
    print(f"  {'ok ' if os.path.exists(p) else '-- '} {k}: {p}")
shutil.copy(f"{SN}/results/SNP_overlap.csv", f"{WORK}/SNP_overlap.published.csv")      # keep the published table for the comparison below

## 7.1 The inputs
50 GTEx tissues, each with a `_pos.vcf` (causal variants) and a `_neg.vcf` (matched non-causal, PIP < 0.01); 26 tissues map to an EpiMap biosample; the analysis keeps the 16 with more than 500 causal variants.

In [ ]:
vcf = sorted(glob.glob(f"{ENHANCER_ROOT}/enformer_comparison/eqtl-enformer/*_pos.vcf")); print(len(vcf), "tissues with causal variants, e.g.", os.path.basename(vcf[0]))
print(open(vcf[0]).read().split("\n")[1][:80], "...")
m = pd.read_csv(f"{ENHANCER_ROOT}/geuvadis/predixscan/en_models_to_BSSID.csv"); m["causal_variants"] = [sum(1 for l in open(f"{ENHANCER_ROOT}/enformer_comparison/eqtl-enformer/{t}_pos.vcf") if not l.startswith("#")) for t in m.PrediXscan_name]
print(m.sort_values("causal_variants", ascending=False).to_string(index=False)); print((m.causal_variants > 500).sum(), "biosamples with > 500 causal variants")

## 7.2 Count the variants at core-motif and control-motif sites
For each biosample: FIMO sites (p < 10⁻⁴) of the 43 core motifs and of each control set in the promoter windows -> genome coordinates of the site centres -> causal variants within ± 4 bp, and the number of motif occurrences. About 8 min.

In [ ]:
run("python snp_overlap_universal.py", cwd=SN)
new = pd.read_csv(f"{SN}/results/SNP_overlap.csv").set_index("bssid"); old = pd.read_csv(f"{WORK}/SNP_overlap.published.csv").set_index("bssid")
cols = [c for c in ("num_pos_SNP", "universal_pos_overlap", "universal_total", "ctrl_mean_pos", "ctrl_mean_total") if c in new.columns]
print(new[cols].to_string()); print("\nidentical to the published table:", new.reindex(old.index)[cols].round(6).equals(old[cols].round(6)))
print(open(f"{SN}/results/wilcoxon_stats.csv").read())

## 7.3 Panels a and b

In [ ]:
run("python -c \"import make_fig4_panels as P; P.panel_4a(); P.panel_4b()\"", cwd=F4)
for p in ("fig4a_eqtl_count", "fig4b_eqtl_ratio"): display(Image(f"{F4}/{p}.png", width=300))